In [1]:
import json
import os
from typing import List, Dict
from datetime import datetime

In [2]:
class ActualReviewerExtractor:
    """Extract actual reviewer assignments from PR data"""
    
    def __init__(self, json_file_path: str):
        self.json_file_path = json_file_path
        self.pr_data = []
        self.actual_assignments = []
    
    def load_pr_data(self):
        """Load PR data from JSON file"""
        print(f"📂 Loading PR data from {self.json_file_path}...")
        
        try:
            with open(self.json_file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract PR reviewer data
            self.pr_data = data.get('pr_reviewer_data', [])
            
            print(f"✅ Loaded {len(self.pr_data)} PRs with review data")
            return True
            
        except Exception as e:
            print(f"❌ Error loading data: {e}")
            return False
    
    def extract_actual_reviewers(self):
        """Extract actual reviewer assignments from each PR"""
        print("🔄 Extracting actual reviewer assignments...")
        
        for pr in self.pr_data:
            pr_number = pr.get('pr_number')
            pr_title = pr.get('title', 'No title')
            pr_author = pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown'
            pr_state = pr.get('state', 'unknown')
            created_at = pr.get('created_at', 'Unknown')
            
            # Extract reviews
            reviews = pr.get('reviews', [])
            actual_reviewers = []
            
            for review in reviews:
                reviewer_info = {
                    'reviewer_username': review.get('reviewer_username', 'Unknown'),
                    'reviewer_id': review.get('reviewer_id'),
                    'review_state': review.get('state', 'unknown'),
                    'review_body': review.get('body', ''),
                    'submitted_at': review.get('submitted_at', 'Unknown'),
                    'author_association': review.get('author_association', 'Unknown')
                }
                actual_reviewers.append(reviewer_info)
            
            # Extract review comments authors
            review_comments = pr.get('review_comments', [])
            comment_authors = set()  # Use set to avoid duplicates
            
            for comment in review_comments:
                if comment.get('user', {}).get('login'):
                    comment_authors.add(comment.get('user', {}).get('login'))
            
            # Create assignment record
            assignment_record = {
                'pr_number': pr_number,
                'pr_title': pr_title,
                'pr_author': pr_author,
                'pr_state': pr_state,
                'created_at': created_at,
                'pr_url': f"https://github.com/expressjs/express/pull/{pr_number}",
                'total_reviews': len(reviews),
                'total_review_comments': len(review_comments),
                'actual_reviewers': actual_reviewers,
                'unique_reviewers': list(set([r['reviewer_username'] for r in actual_reviewers if r['reviewer_username'] != 'Unknown'])),
                'review_comment_authors': list(comment_authors),
                'all_participants': list(set(
                    [r['reviewer_username'] for r in actual_reviewers if r['reviewer_username'] != 'Unknown'] +
                    list(comment_authors)
                ))
            }
            
            self.actual_assignments.append(assignment_record)
        
        print(f"✅ Extracted reviewer data for {len(self.actual_assignments)} PRs")
        return self.actual_assignments
    
    def generate_summary_statistics(self):
        """Generate summary statistics about actual reviewer assignments"""
        print("📊 Generating summary statistics...")
        
        # Count reviewer activity
        reviewer_counts = {}
        reviewer_review_counts = {}
        total_prs_with_reviews = 0
        total_reviews = 0
        
        for assignment in self.actual_assignments:
            if assignment['total_reviews'] > 0:
                total_prs_with_reviews += 1
            
            total_reviews += assignment['total_reviews']
            
            # Count unique reviewers per PR
            for reviewer in assignment['unique_reviewers']:
                reviewer_counts[reviewer] = reviewer_counts.get(reviewer, 0) + 1
            
            # Count total reviews per reviewer
            for review in assignment['actual_reviewers']:
                reviewer = review['reviewer_username']
                if reviewer != 'Unknown':
                    reviewer_review_counts[reviewer] = reviewer_review_counts.get(reviewer, 0) + 1
        
        # Calculate statistics
        avg_reviews_per_pr = total_reviews / len(self.actual_assignments) if self.actual_assignments else 0
        prs_without_reviews = len(self.actual_assignments) - total_prs_with_reviews
        
        summary = {
            'extraction_metadata': {
                'extraction_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'source_file': self.json_file_path,
                'total_prs_analyzed': len(self.actual_assignments)
            },
            'overall_statistics': {
                'total_prs': len(self.actual_assignments),
                'prs_with_reviews': total_prs_with_reviews,
                'prs_without_reviews': prs_without_reviews,
                'total_reviews': total_reviews,
                'average_reviews_per_pr': round(avg_reviews_per_pr, 2),
                'unique_reviewers_count': len(reviewer_counts)
            },
            'reviewer_activity': {
                'by_prs_reviewed': dict(sorted(reviewer_counts.items(), key=lambda x: x[1], reverse=True)),
                'by_total_reviews': dict(sorted(reviewer_review_counts.items(), key=lambda x: x[1], reverse=True))
            },
            'top_reviewers': {
                'most_prs_reviewed': max(reviewer_counts.items(), key=lambda x: x[1]) if reviewer_counts else None,
                'most_total_reviews': max(reviewer_review_counts.items(), key=lambda x: x[1]) if reviewer_review_counts else None
            }
        }
        
        return summary
    
    def save_to_json(self, output_filename: str = "actual_pr_reviewer_assignments.json"):
        """Save the extracted data to JSON file"""
        print(f"💾 Saving data to {output_filename}...")
        
        try:
            # Generate summary statistics
            summary = self.generate_summary_statistics()
            
            # Create final output structure
            output_data = {
                'summary': summary,
                'pr_reviewer_assignments': self.actual_assignments
            }
            
            # Save to file
            with open(output_filename, 'w', encoding='utf-8') as f:
                json.dump(output_data, f, indent=2, ensure_ascii=False)
            
            print(f"✅ Successfully saved {len(self.actual_assignments)} PR assignments to {output_filename}")
            return True
            
        except Exception as e:
            print(f"❌ Error saving to file: {e}")
            return False
    
    def display_summary(self):
        """Display a summary of the extracted data"""
        summary = self.generate_summary_statistics()
        
        print("📋 ACTUAL REVIEWER ASSIGNMENT SUMMARY")
        print("=" * 50)
        
        overall = summary['overall_statistics']
        print(f"📊 Total PRs Analyzed: {overall['total_prs']}")
        print(f"✅ PRs with Reviews: {overall['prs_with_reviews']}")
        print(f"❌ PRs without Reviews: {overall['prs_without_reviews']}")
        print(f"📝 Total Reviews: {overall['total_reviews']}")
        print(f"📈 Average Reviews per PR: {overall['average_reviews_per_pr']}")
        print(f"👥 Unique Reviewers: {overall['unique_reviewers_count']}")
        
        print(f"\n🏆 TOP REVIEWERS BY PRS:")
        top_by_prs = list(summary['reviewer_activity']['by_prs_reviewed'].items())[:10]
        for i, (reviewer, count) in enumerate(top_by_prs, 1):
            print(f"  {i:2d}. {reviewer:20} → {count:2d} PRs")
        
        print(f"\n🎯 TOP REVIEWERS BY TOTAL REVIEWS:")
        top_by_reviews = list(summary['reviewer_activity']['by_total_reviews'].items())[:10]
        for i, (reviewer, count) in enumerate(top_by_reviews, 1):
            print(f"  {i:2d}. {reviewer:20} → {count:2d} reviews")
        
        print("=" * 50)

# Create the extractor instance with correct path (going up one level from PrecisionMetricsCalculation)
print("🚀 Creating Actual Reviewer Extractor...")
extractor = ActualReviewerExtractor("../PR data for Reviewers/moment_moment_reviewer_data_test.json")
print("✅ Extractor ready!")

🚀 Creating Actual Reviewer Extractor...
✅ Extractor ready!


In [3]:
# Load the PR data
success = extractor.load_pr_data()

if success:
    print("🎉 PR data loaded successfully!")
else:
    print("❌ Failed to load PR data. Check file path.")

📂 Loading PR data from ../PR data for Reviewers/moment_moment_reviewer_data_test.json...
✅ Loaded 39 PRs with review data
🎉 PR data loaded successfully!


In [4]:
# Extract actual reviewer assignments
if extractor.pr_data:
    actual_assignments = extractor.extract_actual_reviewers()
    
    # Display summary
    extractor.display_summary()
else:
    print("❌ No PR data available for extraction")

🔄 Extracting actual reviewer assignments...
✅ Extracted reviewer data for 39 PRs
📊 Generating summary statistics...
📋 ACTUAL REVIEWER ASSIGNMENT SUMMARY
📊 Total PRs Analyzed: 39
✅ PRs with Reviews: 39
❌ PRs without Reviews: 0
📝 Total Reviews: 115
📈 Average Reviews per PR: 2.95
👥 Unique Reviewers: 39

🏆 TOP REVIEWERS BY PRS:
   1. anandfresh           → 15 PRs
   2. marwahaha            →  6 PRs
   3. ichernev             →  5 PRs
   4. rksp25               →  3 PRs
   5. ashsearle            →  2 PRs
   6. icambron             →  2 PRs
   7. maggiepint           →  2 PRs
   8. PriyaBihani          →  1 PRs
   9. stephenramthun       →  1 PRs
  10. Spiralis             →  1 PRs

🎯 TOP REVIEWERS BY TOTAL REVIEWS:
   1. anandfresh           → 16 reviews
   2. butterflyhug         → 15 reviews
   3. marwahaha            → 12 reviews
   4. ichernev             →  9 reviews
   5. Spiralis             →  8 reviews
   6. maggiepint           →  7 reviews
   7. ashsearle            →  5 reviews

In [5]:
# Save to JSON file
if extractor.actual_assignments:
    output_file = "actual_pr_reviewer_assignments.json"
    save_success = extractor.save_to_json(output_file)
    
    if save_success:
        print(f"\n🎯 SUCCESS! Actual reviewer assignments saved to '{output_file}'")
        print(f"📊 File contains data for {len(extractor.actual_assignments)} PRs")
        print(f"💾 Includes detailed reviewer information and statistics")
    else:
        print("❌ Failed to save assignments")
else:
    print("❌ No assignments to save")

💾 Saving data to actual_pr_reviewer_assignments.json...
📊 Generating summary statistics...
✅ Successfully saved 39 PR assignments to actual_pr_reviewer_assignments.json

🎯 SUCCESS! Actual reviewer assignments saved to 'actual_pr_reviewer_assignments.json'
📊 File contains data for 39 PRs
💾 Includes detailed reviewer information and statistics
